In [ ]:
from models.ML.ParliamentaryVectorization import ParliamentaryVectorization
from models.ML.PULKMeans import PULKMeans

In [ ]:
subset = "dataset/parcanDeb-mp/150"

In [ ]:
# Asumiendo que tienes el código anterior en el mismo script o importado
# from vectorization_step import ParliamentaryVectorization (o copia la clase anterior aquí)

def run_pul_pipeline(subset_path):
    # 1. Vectorización Global
    pv = ParliamentaryVectorization(subset_path)
    pv.load_and_vectorize()
    
    # 2. Seleccionar un diputado de prueba (el que tenga más intervenciones para que sea interesante)
    # Buscamos el MP con más datos
    target_mp = max(pv.mp_indices, key=lambda k: len(pv.mp_indices[k]))
    print(f"\n--- Probando PUL-KM para el diputado: {target_mp} ---")
    
    # 3. Obtener matrices P y U
    P, U = pv.get_data_for_mp(target_mp)
    
    # 4. Ejecutar algoritmo del Paper
    pul_model = PULKMeans()
    rn_indices = pul_model.fit(P, U)
    
    # 5. Validación de Salida
    # Ahora tenemos:
    # - Positivos (P): Las intervenciones de target_mp
    # - Negativos Fiables (RN): Las filas de U indexadas por rn_indices
    RN_matrix = U[rn_indices]
    
    print(f"\nResumen Dataset Entrenamiento Final para {target_mp}:")
    print(f"Positivos (Clase 1): {P.shape[0]} documentos")
    print(f"Negativos (Clase 0): {RN_matrix.shape[0]} documentos (Negativos Fiables)")
    print(f"Descartados (Ruido): {U.shape[0] - RN_matrix.shape[0]} documentos (Ambiguos)")

In [ ]:
run_pul_pipeline(subset)

In [ ]:
# --- EJECUCIÓN ---
from models.ML.ParliamentaryClassifier import ParliamentaryClassifier

dataset = "dataset/parcanDeb-mp/"
all_subset = ["10", "25", "75", "150"]
for subset in all_subset:
    print(f"\n=== Ejecutando Pipeline Completo para Subconjunto: {subset} ===")
    pipeline = ParliamentaryClassifier(subset_path=dataset + subset)
    resultados = pipeline.run_full_pipeline()

In [ ]:
resultados